과제2-1

가구 제조자의 주당 목재 수요는 각 공장에서 각각 500, 700, 600톤  
가구제조자는 세개의 목재를 세개의 회사에서 구매할 수 있다.  
목재 제조자1,2는 공급량 제약 없다. 목재 제조자3은 주당 500톤초과 공급 불가.  
목재 제조자 2,3은 각 가구별 운송 200톤 제약  

목재-> 가구
x1~3(목재1이)
x4~6(목재2가)
x7~9

x1+x4+x7>=500(가구1)
x2+x5+x8>=700(가구2)
x3+x6+x9>=600(가구3)
x7+x8+x9<=500(목재3)
x4<=200, x5<=200, x6<=200
x7<=200, x8<=200, x9<=200

min(2x1+3x2+5x3
+2.5x4+4x5+4.8x6
+3x7+3.6x8+3.9x9)



In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

# 대응시간(분)
T = [
    [5, 12, 30, 20, 15],
    [20, 4, 15, 10, 25],
    [15, 20, 6, 15, 12],
    [25, 15, 25, 4, 10],
    [10, 25, 15, 12, 5]
]

# 1일 평균 화재 횟수
F = [2, 1, 3, 1, 3]

n = 5

# 의사결정변수
# x[i] = 구역 i에 소방서를 설치하면 1, 아니면 0
x = {}
for i in range(n):
    x[i] = solver.IntVar(0, 1, "x[%i]" % i)

# y[i,j] = 구역 j의 화재를 구역 i의 소방서가 담당하면 1, 아니면 0
y = {}
for i in range(n):
    for j in range(n):
        y[i, j] = solver.IntVar(0, 1, "y[%i][%i]" % (i, j))

# 제약조건
# 소방서는 정확히 2개 설치
solver.Add(sum(x[i] for i in range(n)) == 2, "two_station")

# 각 구역의 화재는 정확히 1개의 소방서에 의해 담당
for j in range(n):
    solver.Add(sum(y[i, j] for i in range(n)) == 1, "assign_st_%i" % j)

# 소방서가 설치된 구역만 담당 가능
for i in range(n):
    for j in range(n):
        solver.Add(y[i, j] <= x[i], "st배치보장_%i_%i" % (i, j))

# 목적함수: 평균 화재 횟수를 가중치로 둔 총 대응시간 최소화
obj_expr = []
for i in range(n):
    for j in range(n):
        obj_expr.append(F[j] * T[i][j] * y[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or_assignment_firestation.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())

    print("\n[소방서 설치 구역]")
    for i in range(n):
        print(x[i].name(), "=", x[i].solution_value())

    print("\n[구역별 담당 소방서]")
    for j in range(n):
        for i in range(n):
            if y[i, j].solution_value() == 1:
                print("구역 [%i] --> 소방서 [%i]" % (j + 1, i + 1))
else:
    print("The problem does not have an optimal solution.")

가구 제조자의 주당 목재 수요는 각 공장에서 각각 500, 700, 600톤  
가구제조자는 세개의 목재를 세개의 회사에서 구매할 수 있다.  
목재 제조자1,2는 공급량 제약 없다. 목재 제조자3은 주당 500톤초과 공급 불가.  
목재 제조자 2,3은 각 가구별 운송 200톤 제약  

목재-> 가구
x1~3(목재1이)
x4~6(목재2가)
x7~9

x1+x4+x7>=500(가구1)
x2+x5+x8>=700(가구2)
x3+x6+x9>=600(가구3)
x7+x8+x9<=500(목재3)
x4<=200, x5<=200, x6<=200
x7<=200, x8<=200, x9<=200

min(2x1+3x2+5x3
+2.5x4+4x5+4.8x6
+3x7+3.6x8+3.9x9)

In [2]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

# 비용($ per ton)
C = [
    [2, 3, 5],
    [2.5, 4, 4.8],
    [3, 3.6, 3.2]
]

# 주당 목재 수요
D = [500, 700, 600]

n = 3

# 의사결정변수
# y[i,j] = 목재 회사 i가 가구 공장 j에게
x = {}
for i in range(n):
    for j in range(n):
        x[i, j] = solver.IntVar(0, solver.infinity(), "x[%i][%i]" % (i, j))

# 제약조건
# 가구의 주당 수요
for j in range(n):
    solver.Add(sum(x[i, j] for i in range(n)) == D[j], "가구 회사 수요")

# 목재3 공급 제한
solver.Add(sum(x[2, j] for j in range(n)) <= 500, "목재3 공급 제한")

# 가구별 운송 제약
for i in range(1,3):
    for j in range(n):
        solver.Add(x[i, j] <= 200, "가구 운송 제약")

obj_expr = []
for i in range(n):
    for j in range(n):
        obj_expr.append(C[i][j] * x[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or_1.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())

    print("\n[목재별 가구회사]")
    for i in range(n):
        for j in range(n):
            print("목재 [%i] --> 가구 [%i]" % (i + 1, j + 1),
            " : ", x[i, j].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  5700.0

[목재별 가구회사]
목재 [1] --> 가구 [1]  :  500.0
목재 [1] --> 가구 [2]  :  700.0
목재 [1] --> 가구 [3]  :  200.0
목재 [2] --> 가구 [1]  :  0.0
목재 [2] --> 가구 [2]  :  0.0
목재 [2] --> 가구 [3]  :  200.0
목재 [3] --> 가구 [1]  :  0.0
목재 [3] --> 가구 [2]  :  0.0
목재 [3] --> 가구 [3]  :  200.0
